In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.interpolate import CubicSpline, PchipInterpolator

from statsmodels.robust.scale import mad

from sklearn.gaussian_process.kernels import RBF, WhiteKernel, RationalQuadratic, ConstantKernel as C
from sklearn.gaussian_process import GaussianProcessRegressor

from skimage.restoration import denoise_tv_chambolle

from Kakapo.photometry import forced_photometry

%matplotlib widget

In [ ]:
def local_mad(y, halfwin=25):
    # halfwin in number of cadences; tune ~ hours in K2 cadence
    n = len(y)
    out = np.empty(n)
    for i in range(n):
        lo = max(0, i-halfwin); hi = min(n, i+halfwin+1)
        out[i] = 1.4826 * np.nanmedian(np.abs(y[lo:hi] - np.nanmedian(y[lo:hi])))
    return np.maximum(out, np.nanpercentile(out, 5))  # avoid zeros

def ew_mad(y, alpha=0.01):
    # crude exponentially weighted MAD as a slow noise floor proxy
    med, madv = 0.0, 1.0
    out = np.empty_like(y, dtype=float)
    for i, v in enumerate(y):
        med = (1-alpha)*med + alpha*v
        madv = (1-alpha)*madv + alpha*abs(v - med)
        out[i] = 1.4826 * madv
    return out


In [ ]:
def alpha_time_linear(t, base, slope, t0=None):
    t = np.asarray(t)
    if t0 is None:
        t0 = t.min()
    ramp = base + slope * (t - t0)
    return np.clip(ramp, a_min=base, a_max=None)

def alpha_time_exp(t, base, amp, tau, t0=None):
    t = np.asarray(t)
    if t0 is None: t0 = t.min()
    return base + amp * (1.0 - np.exp(-(t - t0)/tau))

def alpha_time_binned(t, flux_err=None, n_bins=20, floor=0.0):
    t = np.asarray(t)
    bins = np.linspace(t.min(), t.max(), n_bins+1)
    idx = np.digitize(t, bins) - 1
    idx = np.clip(idx, 0, n_bins-1)
    if flux_err is None:
        # fallback: monotone floor if no per-point errors
        vals = floor + (np.arange(n_bins) / max(1, n_bins-1)) * floor
    else:
        per_bin = np.full(n_bins, np.nan)
        for k in range(n_bins):
            mask = (idx == k)
            if np.any(mask):
                per_bin[k] = np.nanmedian(flux_err[mask]**2)
        # fill empties, enforce monotonic non-decreasing if desired
        # (monotone smoothing prevents oscillations)
        # simple forward fill:
        last = np.nan
        for k in range(n_bins):
            if np.isnan(per_bin[k]):
                per_bin[k] = last if not np.isnan(last) else np.nanmedian(per_bin)
            last = per_bin[k]
        # optional: make it non-decreasing
        per_bin = np.maximum.accumulate(per_bin)
        vals = per_bin
    return vals[idx] + floor


In [ ]:
def gauss_smooth(time, flux, flux_error=None, n_samples=11):
    kernel = (C(0.1) * RBF(length_scale=3.0) +
              C(1e-4, (5e-5, 5e-3)) * RBF(length_scale=0.25, length_scale_bounds=(0.05, 0.5)) +
              WhiteKernel(noise_level=3e-3))

    base_alpha_values = [0.05, 0.15, 0.25]
    weights_norm = 1

    for base_alpha in base_alpha_values:
        try:
            
            alpha_per_point = (base_alpha / weights_norm) ** 2
            if flux_error is not None:
                # max_var = np.nanpercentile(flux_error**2, 95)  # or a fixed ceiling
                # alpha = np.minimum(flux_error**2, max_var) + alpha_per_point
                alpha = (flux_error**2) + alpha_per_point
            else:
                alpha = alpha_per_point

            gp = GaussianProcessRegressor(kernel=kernel,
                                          alpha=alpha,
                                          normalize_y=True,
                                          optimizer=None)
            gp.fit(time[:, None], flux)

            samples = gp.sample_y(time[:, None], n_samples=n_samples) # GP draws
            sampled_mean = samples.mean(axis=1)
            sampled_std  = samples.std(axis=1)

           
            _, gp_std = gp.predict(time[:, None], return_std=True) # GP predictive std

            residuals = flux - sampled_mean # Residual scatter (MAD-based, robust)
            local_scatter = mad(residuals)

            total_std = np.sqrt(gp_std**2 + sampled_std**2 + local_scatter**2) # Final error = quadrature of GP std, sample scatter, and residual scatter

            return sampled_mean, 2 * total_std  # mean + error band
        except Exception:
            continue

    mean_pred = PchipInterpolator(time, flux)(time)
    std_pred = np.full_like(mean_pred, np.nanstd(flux))
    return mean_pred, std_pred

In [ ]:
def gauss_smooth_MAD(time, flux, flux_error=None, n_samples=11):
    kernel = (C(0.1) * RBF(length_scale=3.0) +
              C(1e-4, (5e-5, 5e-3)) * RBF(length_scale=0.25, length_scale_bounds=(0.05, 0.5)) +
              WhiteKernel(noise_level=3e-3))

    base_alpha_values = [0.05, 0.15, 0.25]
    weights_norm = 1

    for base_alpha in base_alpha_values:
        try:
            
            alpha_per_point = (base_alpha / weights_norm) ** 2
            if flux_error is not None:
                window = int(0.02 * len(time))  # ~5% of LC length, tunable
                local_mad = np.array([mad(flux[max(0,i-window):min(len(flux), i+window)]) for i in range(len(flux))])
                alpha = flux_error**2 + (local_mad**2 + alpha_per_point)
            else:
                alpha = alpha_per_point
                


            gp = GaussianProcessRegressor(kernel=kernel,
                                          alpha=alpha,
                                          normalize_y=True,
                                          optimizer=None)
            gp.fit(time[:, None], flux)

            samples = gp.sample_y(time[:, None], n_samples=n_samples) # GP draws
            sampled_mean = samples.mean(axis=1)
            sampled_std  = samples.std(axis=1)

           
            _, gp_std = gp.predict(time[:, None], return_std=True) # GP predictive std

            residuals = flux - sampled_mean # Residual scatter (MAD-based, robust)
            local_scatter = mad(residuals)

            total_std = np.sqrt(gp_std**2 + sampled_std**2 + local_scatter**2) # Final error = quadrature of GP std, sample scatter, and residual scatter

            return sampled_mean, 2 * total_std  # mean + error band
        except Exception:
            continue

    mean_pred = PchipInterpolator(time, flux)(time)
    std_pred = np.full_like(mean_pred, np.nanstd(flux))
    return mean_pred, std_pred

In [ ]:
def gauss_smooth_msf(time, flux, flux_error=None, n_samples=11):

    kernel = (C(0.1) * RBF(length_scale=3.0) +
            RationalQuadratic(alpha=1.0, length_scale=1.0) +  
            WhiteKernel(noise_level=3e-3))


    base_alpha_values = [0.05, 0.15, 0.25]
    weights_norm = 1

    for base_alpha in base_alpha_values:
        try:
            
            alpha_per_point = (base_alpha / weights_norm) ** 2
            if flux_error is not None:
                # max_var = np.nanpercentile(flux_error**2, 95)  # or a fixed ceiling
                # alpha = np.minimum(flux_error**2, max_var) + alpha_per_point
                alpha = (flux_error**2) + alpha_per_point
            else:
                alpha = alpha_per_point

            gp = GaussianProcessRegressor(kernel=kernel,
                                          alpha=alpha,
                                          normalize_y=True,
                                          optimizer=None)
            gp.fit(time[:, None], flux)

            samples = gp.sample_y(time[:, None], n_samples=n_samples) # GP draws
            sampled_mean = samples.mean(axis=1)
            sampled_std  = samples.std(axis=1)

           
            _, gp_std = gp.predict(time[:, None], return_std=True) # GP predictive std

            residuals = flux - sampled_mean # Residual scatter (MAD-based, robust)
            local_scatter = mad(residuals)

            total_std = np.sqrt(gp_std**2 + sampled_std**2 + local_scatter**2) # Final error = quadrature of GP std, sample scatter, and residual scatter

            return sampled_mean, 2 * total_std  # mean + error band
        except Exception:
            continue
    
    mean_pred = PchipInterpolator(time, flux)(time)
    std_pred = np.full_like(mean_pred, np.nanstd(flux))
    return mean_pred, std_pred

In [ ]:
def local_mad(y, halfwin=25):
    # halfwin in number of cadences; tune ~ hours in K2 cadence
    n = len(y)
    out = np.empty(n)
    for i in range(n):
        lo = max(0, i-halfwin); hi = min(n, i+halfwin+1)
        out[i] = 1.4826 * np.median(np.abs(y[lo:hi] - np.median(y[lo:hi])))
    return np.maximum(out, np.percentile(out, 5))  # avoid zeros

def ew_mad(y, alpha=0.01):
    # crude exponentially weighted MAD as a slow noise floor proxy
    med, madv = 0.0, 1.0
    out = np.empty_like(y, dtype=float)
    for i, v in enumerate(y):
        med = (1-alpha)*med + alpha*v
        madv = (1-alpha)*madv + alpha*abs(v - med)
        out[i] = 1.4826 * madv
    return out

def gauss_smooth_adaptive(time, flux, flux_error=None, n_samples=11,
                          halfwin=25, motion_proxy=None):

    t = np.asarray(time)
    y = np.asarray(flux)
    yerr = np.asarray(flux_error) if flux_error is not None else None

    # Multi-scale kernel
    kernel = (C(0.1) * RBF(length_scale=3.0) +
              RationalQuadratic(alpha=1.0, length_scale=1.0) +
              WhiteKernel(noise_level=3e-3))

    # Local scatter & slow thermal floor
    mad_loc = local_mad(y, halfwin=halfwin)
    g_slow = ew_mad(y, alpha=0.005)   # slow drift of noise floor

    if motion_proxy is None:
        motion_proxy = np.zeros_like(y)
    # normalize motion to median level
    mp = motion_proxy / (np.median(np.abs(motion_proxy)) + 1e-12)

    # Combine heteroscedastic alpha
    c_mad, c_motion, c_therm, c_min = 1.0, 1.0, 1.0, 1e-3
    alpha = ( (yerr**2 if yerr is not None else 0.0)
              + (c_mad * mad_loc)**2
              + (c_motion * mp)**2
              + (c_therm * g_slow)**2
              + c_min**2 )

    gp = GaussianProcessRegressor(kernel=kernel,
                                  alpha=alpha,
                                  normalize_y=True,
                                  optimizer=None)
    gp.fit(t[:, None], y)

    # Draw samples + predictive std
    samples = gp.sample_y(t[:, None], n_samples=n_samples)
    sampled_mean = samples.mean(axis=1)
    sampled_std  = samples.std(axis=1)
    _, gp_std = gp.predict(t[:, None], return_std=True)

    # Robust residual scatter
    res = y - sampled_mean
    res_mad = local_mad(res, halfwin=halfwin)
    total_std = np.sqrt(gp_std**2 + sampled_std**2 + res_mad**2)

    # Optional baseline re-shrink (tighten where really quiet)
    quiet = (np.abs(res) < 2.5*res_mad) & (np.abs(mp) < 2.0)
    total_std[quiet] = np.minimum(total_std[quiet], 1.2*res_mad[quiet])

    return sampled_mean, 2*total_std


In [ ]:
def gauss_smooth_time_alpha(time, flux, flux_error=None, n_samples=11,
                            alpha_mode='linear',     # 'linear' | 'exp' | 'binned'
                            alpha_params=None,       # dict of params per mode
                            keep_white=True):
    t = np.asarray(time)
    y = np.asarray(flux)
    yerr = np.asarray(flux_error) if flux_error is not None else None

    # --- Kernel: keep your original short/long RBF mix, plus optional WhiteKernel
    kernel = (C(0.1) * RBF(length_scale=3.0) +
              C(1e-4) * RBF(length_scale=0.25))
    if keep_white:
        kernel += WhiteKernel(noise_level=3e-3)

    # --- Build alpha(t)
    if alpha_params is None:
        alpha_params = {}
    if alpha_mode == 'linear':
        base  = alpha_params.get('base',  (np.nanmedian(yerr)**2 if yerr is not None else 1e-6))
        slope = alpha_params.get('slope', base * 0.1 / (t.max()-t.min()+1e-12))
        alpha_t = alpha_time_linear(t, base=base, slope=slope, t0=t.min())
    elif alpha_mode == 'exp':
        base = alpha_params.get('base', (np.nanmedian(yerr)**2 if yerr is not None else 1e-6))
        amp  = alpha_params.get('amp',  base)
        tau  = alpha_params.get('tau',  0.25*(t.max()-t.min()))
        alpha_t = alpha_time_exp(t, base=base, amp=amp, tau=tau, t0=t.min())
    elif alpha_mode == 'binned':
        floor = alpha_params.get('floor', 1e-6)
        alpha_t = alpha_time_binned(t, flux_err=yerr, n_bins=alpha_params.get('n_bins', 20), floor=floor)
    else:
        raise ValueError("alpha_mode must be 'linear', 'exp', or 'binned'")

    # Combine with measurement variance if present
    if yerr is not None:
        alpha_vec = (yerr**2) + alpha_t
    else:
        alpha_vec = alpha_t

    gp = GaussianProcessRegressor(kernel=kernel,
                                  alpha=alpha_vec,
                                  normalize_y=True,
                                  optimizer=None)
    try:
        gp.fit(t[:, None], y)
        samples = gp.sample_y(t[:, None], n_samples=n_samples)
        sampled_mean = samples.mean(axis=1)
        sampled_std  = samples.std(axis=1)
        _, gp_std = gp.predict(t[:, None], return_std=True)
        # Conservative total error (no MAD feedback since that hurt you):
        total_std = np.sqrt(gp_std**2 + sampled_std**2)

        return sampled_mean, 2*total_std
    except Exception:
        # safe fallback
        mean_pred = PchipInterpolator(t, y)(t)
        std_pred = np.full_like(mean_pred, np.nanstd(y))
        return mean_pred, std_pred


In [ ]:
lc_time = np.load('/Users/zgl12/Modules/Kakapo/lc_time.npy')
lc_flux = np.load('/Users/zgl12/Modules/Kakapo/lc_flux.npy')
lc_flux_err = np.load('/Users/zgl12/Modules/Kakapo/lc_flux_err.npy')

mask = np.isfinite(lc_flux) & np.isfinite(lc_time) & np.isfinite(lc_flux_err)

masked_lc_time = lc_time[mask]
masked_lc_flux = lc_flux[mask]
masked_lc_flux_err = lc_flux_err[mask]

print('STD WGT')
smoothed_lc_wgt, smooth_err_wgt = gauss_smooth(masked_lc_time, masked_lc_flux, masked_lc_flux_err)
baseline_01 = denoise_tv_chambolle(masked_lc_flux, weight=0.1)
baseline_001 = denoise_tv_chambolle(masked_lc_flux, weight=0.01)
baseline_1 = denoise_tv_chambolle(masked_lc_flux, weight=1)
baseline_01_sm = denoise_tv_chambolle(smoothed_lc_wgt, weight=0.1)
# print('STD')
# smoothed_lc, smooth_err = gauss_smooth(masked_lc_time, masked_lc_flux, None)
# print('MAD')
# smoothed_lc_MAD, smooth_err_MAD = gauss_smooth_MAD(masked_lc_time, masked_lc_flux, masked_lc_flux_err)
# print('MSF')
# smoothed_lc_msf, smooth_err_msf = gauss_smooth_msf(masked_lc_time, masked_lc_flux, None)
# print('MSF WGT')
# smoothed_lc_msf_wgt, smooth_err_msf_wgt = gauss_smooth_msf(masked_lc_time, masked_lc_flux, masked_lc_flux_err)

In [ ]:
smoothed_lc_msf_ada, smooth_err_msf_ada = gauss_smooth_adaptive(masked_lc_time, masked_lc_flux, masked_lc_flux_err)

In [ ]:
smoothed_lc_alin, smooth_err_alin = gauss_smooth_time_alpha(masked_lc_time, masked_lc_flux, 
                                                            flux_error=masked_lc_flux_err, 
                                                            n_samples=11,
                                                            alpha_mode='linear',     # 'linear' | 'exp' | 'binned'
                                                            alpha_params=None,       # dict of params per mode
                                                            keep_white=True)

In [ ]:
smoothed_lc_exp, smooth_err_exp = gauss_smooth_time_alpha(masked_lc_time, masked_lc_flux, 
                                                          flux_error=masked_lc_flux_err, 
                                                          n_samples=11,
                                                          alpha_mode='exp',     # 'linear' | 'exp' | 'binned'
                                                          alpha_params=None,       # dict of params per mode
                                                          keep_white=True)

In [ ]:
smoothed_lc_bin, smooth_err_bin = gauss_smooth_time_alpha(masked_lc_time, masked_lc_flux, 
                                                          flux_error=masked_lc_flux_err, 
                                                          n_samples=11,
                                                          alpha_mode='binned',     # 'linear' | 'exp' | 'binned'
                                                          alpha_params=None,       # dict of params per mode
                                                          keep_white=True)

In [ ]:
baseline_1_sm = denoise_tv_chambolle(smoothed_lc_wgt, weight=0.5)

In [ ]:
from scipy.interpolate import UnivariateSpline

def alpha_from_spline(time, flux_err, spline_times, spline_values,
                      mode='variance', c_therm=1.0, floor=0.0):
    # Evaluate your background spline B(t) on LC times
    B = UnivariateSpline(spline_times, spline_values, k=3, s=0)(time)
    B = np.maximum(B, np.nanpercentile(B, 5))  # avoid zeros/negatives

    if mode == 'variance':
        # Assume var ~ flux_err^2 + c*(B/B0)
        B0 = np.median(B)
        therm_var = c_therm * (B / B0)
    elif mode == 'std':
        # Assume std ~ sqrt(flux_err^2 + (c*B)^2)
        therm_var = (c_therm * B)**2
    else:
        raise ValueError("mode must be 'variance' or 'std'")

    return (flux_err**2) + therm_var + floor


def gp_with_spline_mean(time, flux, flux_err, spline_times, spline_values,
                        kernel=None, n_samples=11):
    t = np.asarray(time)
    y = np.asarray(flux)
    yerr = np.asarray(flux_err)

    # 1) Fit (or pass) a cubic spline that models the *additive* drift
    # If you already have the modelled spline, just evaluate it:
    drift_spline = UnivariateSpline(spline_times, spline_values, k=3, s=0)
    m_t = drift_spline(t)

    # Optional: scale coupling if needed (linear fit y ~ a*m_t + b)
    A = np.vstack([m_t, np.ones_like(m_t)]).T
    a, b = np.linalg.lstsq(A, y, rcond=None)[0]
    mean_trend = a*m_t + b

    # 2) Detrend, then GP on residuals
    y_res = y - mean_trend

    if kernel is None:
        kernel = (C(0.1)*RBF(3.0) + C(1e-3)*RBF(0.25) + WhiteKernel(3e-3))

    gp = GaussianProcessRegressor(kernel=kernel, alpha=yerr**2,
                                  normalize_y=True, optimizer=None)
    gp.fit(t[:,None], y_res)

    pred_res, pred_std = gp.predict(t[:,None], return_std=True)

    # 3) Add mean back
    pred = pred_res + mean_trend
    err  = 2*pred_std
    
def gauss_smooth_cubic(time, flux, flux_error=None, n_samples=11):
    kernel = (C(0.1) * RBF(length_scale=3.0) +
              C(1e-4, (5e-5, 5e-3)) * RBF(length_scale=0.25, length_scale_bounds=(0.05, 0.5)) +
              WhiteKernel(noise_level=3e-3))

    base_alpha_values = [0.05, 0.15, 0.25]
    weights_norm = 1

    for base_alpha in base_alpha_values:
        try:
            
            alpha_per_point = (base_alpha / weights_norm) ** 2
            if flux_error is not None:
                # max_var = np.nanpercentile(flux_error**2, 95)  # or a fixed ceiling
                # alpha = np.minimum(flux_error**2, max_var) + alpha_per_point
                alpha = (flux_error**2) + alpha_per_point
            else:
                alpha = alpha_per_point

            gp = GaussianProcessRegressor(kernel=kernel,
                                          alpha=alpha,
                                          normalize_y=True,
                                          optimizer=None)
            gp.fit(time[:, None], flux)

            samples = gp.sample_y(time[:, None], n_samples=n_samples) # GP draws
            sampled_mean = samples.mean(axis=1)
            sampled_std  = samples.std(axis=1)

           
            _, gp_std = gp.predict(time[:, None], return_std=True) # GP predictive std

            residuals = flux - sampled_mean # Residual scatter (MAD-based, robust)
            local_scatter = mad(residuals)

            total_std = np.sqrt(gp_std**2 + sampled_std**2 + local_scatter**2) # Final error = quadrature of GP std, sample scatter, and residual scatter

            return sampled_mean, 2 * total_std  # mean + error band
        except Exception:
            continue

    mean_pred = PchipInterpolator(time, flux)(time)
    std_pred = np.full_like(mean_pred, np.nanstd(flux))
    return mean_pred, std_pred

In [ ]:
plt.figure()
# plt.scatter(masked_lc_time, masked_lc_flux, alpha = 0.05, zorder = 10)
plt.scatter(masked_lc_time, smoothed_lc_wgt, alpha = 0.05)
# plt.scatter(masked_lc_time, baseline_001, alpha = 0.05)
# plt.scatter(masked_lc_time, baseline_01, alpha = 0.05)
# plt.scatter(masked_lc_time, baseline_1, alpha = 0.05)
# plt.scatter(masked_lc_time, baseline_01_sm, alpha = 0.05, s = 10)
# plt.scatter(masked_lc_time, baseline_001_sm, alpha = 0.05, s = 10)
plt.scatter(masked_lc_time, baseline_1_sm, alpha = 0.05, s = 10)
# plt.scatter(masked_lc_time, smoothed_lc, alpha = 0.05)
# plt.scatter(masked_lc_time, smoothed_lc_MAD, alpha = 0.05)
# plt.scatter(masked_lc_time, smoothed_lc_alin, alpha = 0.05)
# plt.scatter(masked_lc_time, smoothed_lc_exp, alpha = 0.05)
# plt.scatter(masked_lc_time, smoothed_lc_bin, alpha = 0.05)
# plt.scatter(masked_lc_time, smoothed_lc_msf_wgt, alpha = 0.05)
# plt.scatter(masked_lc_time, smoothed_lc_msf, alpha = 0.05)
# plt.scatter(masked_lc_time, smoothed_lc_msf_ada, alpha = 0.05)
plt.show()

In [ ]:
diff_file = '/Users/zgl12/Modules/Kakapo/Data/difference_arrays/c3/diff_c3_t205922648.npy'

diff = np.load(diff_file)

new_diffs = []

for i in range(len(diff)):
    new_diff = denoise_tv_chambolle(diff[i], weight=0.1)
    new_diffs.append(new_diff)
    
new_diffs = np.array(new_diffs)

In [ ]:
new_flux = forced_photometry(new_diffs, 7.36, 6.15, None)
new_mask = np.isfinite(new_flux) & np.isfinite(lc_time) & np.isfinite(lc_flux_err)

In [ ]:
smoothed_new, smooth_err_new = gauss_smooth(lc_time[new_mask], new_flux[new_mask], lc_flux_err[new_mask])

In [ ]:
np.nansum(np.isnan(new_flux))

In [ ]:
plt.figure()
plt.scatter(masked_lc_time, masked_lc_flux, alpha = 0.05, zorder = 10)
plt.scatter(masked_lc_time, smoothed_lc_wgt, alpha = 0.05)
# plt.scatter(masked_lc_time, baseline_001, alpha = 0.05)
# plt.scatter(masked_lc_time, baseline_01, alpha = 0.05)
# plt.scatter(masked_lc_time, baseline_1, alpha = 0.05)
# plt.scatter(masked_lc_time, baseline_01_sm, alpha = 0.05, s = 10)
# plt.scatter(masked_lc_time, baseline_001_sm, alpha = 0.05, s = 10)
# plt.scatter(masked_lc_time, baseline_1_sm, alpha = 0.05, s = 10)
plt.scatter(lc_time[new_mask], smoothed_new, alpha = 0.05)
# plt.scatter(masked_lc_time, smoothed_lc, alpha = 0.05)
# plt.scatter(masked_lc_time, smoothed_lc_MAD, alpha = 0.05)
# plt.scatter(masked_lc_time, smoothed_lc_alin, alpha = 0.05)
# plt.scatter(masked_lc_time, smoothed_lc_exp, alpha = 0.05)
# plt.scatter(masked_lc_time, smoothed_lc_bin, alpha = 0.05)
# plt.scatter(masked_lc_time, smoothed_lc_msf_wgt, alpha = 0.05)
# plt.scatter(masked_lc_time, smoothed_lc_msf, alpha = 0.05)
# plt.scatter(masked_lc_time, smoothed_lc_msf_ada, alpha = 0.05)
plt.show()

In [ ]:
def angular_distance(ra1, dec1, ra2, dec2):
    """
    Vectorized great-circle distance using haversine formula.
    All inputs assumed in degrees.
    """
    ra1_rad, dec1_rad = np.radians(ra1), np.radians(dec1)
    ra2_rad, dec2_rad = np.radians(ra2), np.radians(dec2)

    delta_ra = ra2_rad - ra1_rad
    delta_dec = dec2_rad - dec1_rad

    a = np.sin(delta_dec / 2.0) ** 2 + np.cos(dec1_rad) * np.cos(dec2_rad) * np.sin(delta_ra / 2.0) ** 2
    return np.degrees(2 * np.arcsin(np.sqrt(a)))

In [ ]:
angular_distance(70.68, -67.94, 130.8129, -12.2294)/5.770

In [ ]:
import numpy as np
from scipy.stats import norm

def sigma_to_confidence(sigma, two_tailed=True):
    """
    Convert sigma significance to confidence level.

    Parameters
    ----------
    sigma : float
        The number of standard deviations (σ).
    two_tailed : bool, optional
        If True, return the two-tailed confidence level (default).
        If False, return the one-tailed.

    Returns
    -------
    confidence : float
        Confidence level between 0 and 1.
    """
    if two_tailed:
        # Two-tailed: P(|Z| < sigma)
        return 2 * norm.cdf(sigma) - 1
    else:
        # One-tailed: P(Z < sigma)
        return norm.cdf(sigma)

# Examples
print(sigma_to_confidence(1))      # ~0.6827 (1σ two-tailed)
print(sigma_to_confidence(2))      # ~0.9545 (2σ two-tailed)
print(sigma_to_confidence(3))      # ~0.9973 (3σ two-tailed)
print(sigma_to_confidence(5))      # ~0.9999994 (5σ two-tailed)

print(sigma_to_confidence(3, two_tailed=False))  # ~0.99865 (3σ one-tailed)


In [ ]:
print(sigma_to_confidence(11.7, two_tailed=False))      # ~0.9999994 (5σ two-tailed)